# 03 — 技能（Skills）

**来源:** [LangChain Docs — Skills](https://docs.langchain.com/oss/python/deepagents/skills)

Skills 是可复用的 Agent 能力，提供专业化工作流和领域知识。遵循 Agent Skills 规范，支持解释器技能（interpreter skills）——提供可导入的函数供解释器调用。

## 1. 什么是 Skills

Skills 是一个文件夹目录，每个子文件夹包含：

| 文件 | 必需 | 说明 |
|------|------|------|
| `SKILL.md` | 是 | 包含 frontmatter 元数据和指令 |
| 脚本文件（.py/.ts） | 否 | 供解释器调用的代码 |
| 参考文档 | 否 | 额外的参考信息 |
| 模板/资源 | 否 | 模板和其他资源 |

> 所有附加资源必须在 `SKILL.md` 中引用，说明文件内容和使用方法。

## 2. Skills 工作原理

### 渐进式披露（Progressive Disclosure）

1. Agent 启动时，读取每个 skill 的 `SKILL.md` **frontmatter**（仅名称和描述）
2. Agent 收到 prompt 后，检查是否有匹配的 skill
3. 找到匹配 skill 后，加载**完整文件内容**

### 目录结构示例

```
skills/
  ├── langgraph-docs/
  │   └── SKILL.md
  └── arxiv_search/
      ├── SKILL.md
      └── arxiv_search.py   # 搜索 arXiv 的代码
```

## 3. SKILL.md 结构

SKILL.md 包含 frontmatter（YAML 元数据）和 Markdown 指令正文。

In [ ]:
from dotenv import load_dotenv
load_dotenv()
# SKILL.md 的完整结构示例（作为字符串展示）

skill_md_example = """---
name: langgraph-docs
description: Use this skill for requests related to LangGraph in order to fetch relevant documentation to provide accurate, up-to-date guidance.
license: MIT
compatibility: Requires internet access for fetching documentation URLs
metadata:
  author: langchain
  version: "1.0"
allowed-tools: fetch_url
module: index.ts
---

# langgraph-docs

## Overview

This skill explains how to access LangGraph Python documentation to help answer questions and guide implementation.

## Instructions

### 1. Fetch the documentation index

Use the fetch_url tool to read the following URL:
https://docs.langchain.com/llms.txt

### 2. Select relevant documentation

Based on the question, identify 2-4 most relevant documentation URLs.

### 3. Fetch selected documentation

Use the fetch_url tool to read the selected URLs.

### 4. Provide accurate guidance

Give a direct answer first. Include key steps or API names.
Avoid quoting long passages. Paraphrase and link instead.

### 5. Provide reference links

Include a **References** section listing the page URLs used.
"""

print(skill_md_example[:200] + "...")  # 仅预览前 200 字符


### Frontmatter 字段说明

| 字段 | 说明 |
|------|------|
| `name` | Skill 名称 |
| `description` | 描述（超过 1024 字符会被截断） |
| `license` | 许可证（可选） |
| `compatibility` | 兼容性说明（可选） |
| `metadata` | 自定义元数据（可选） |
| `allowed-tools` | 允许的工具列表（可选） |
| `module` | 入口模块文件（可选） |

> Deep Agents 中 SKILL.md 文件必须小于 10 MB，超出会被跳过。

## 4. 使用 Skills：三种 Backend 方式

### 4.1 StateBackend（状态内文件系统）

通过 `files` 参数在 `invoke()` 时播种 skill 文件。

In [ ]:
# StateBackend 方式：通过 invoke 的 files 参数播种 skill

from urllib.request import urlopen

from deepagents import create_deep_agent
from deepagents.backends import StateBackend
from deepagents.backends.utils import create_file_data
from langchain_quickjs import CodeInterpreterMiddleware
from langgraph.checkpoint.memory import MemorySaver

checkpointer = MemorySaver()
backend = StateBackend()

# 从 GitHub 下载示例 skill 文件
skill_url = "https://raw.githubusercontent.com/langchain-ai/deepagents/refs/heads/main/libs/cli/examples/skills/langgraph-docs/SKILL.md"
with urlopen(skill_url) as response:
    skill_content = response.read().decode('utf-8')

skills_files = {
    "/skills/langgraph-docs/SKILL.md": create_file_data(skill_content),
}

agent = create_deep_agent(
    model="deepseek-v4-flash",
    backend=backend,
    skills=["/skills/"],
    checkpointer=checkpointer,
    middleware=[CodeInterpreterMiddleware(skills_backend=backend)],  # 解释器技能需要
)

result = agent.invoke(
    {
        "messages": [{"role": "user", "content": "LangGraph 是什么？"}],
        "files": skills_files,  # 播种虚拟文件系统中的技能文件
    },
    config={"configurable": {"thread_id": "12345"}},
)

print(result["messages"][-1].content[:300])

### 4.2 StoreBackend（持久化存储）

通过 `langgraph.store` 持久化存储技能文件。

In [ ]:
# StoreBackend 方式：通过 langgraph.store 持久化 skill

from urllib.request import urlopen

from deepagents import create_deep_agent
from deepagents.backends import StoreBackend
from deepagents.backends.utils import create_file_data
from langchain_quickjs import CodeInterpreterMiddleware
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()
backend = StoreBackend(namespace=lambda _rt: ("filesystem",))

# 从 GitHub 下载 skill 并存入 Store
skill_url = "https://raw.githubusercontent.com/langchain-ai/deepagents/refs/heads/main/libs/cli/examples/skills/langgraph-docs/SKILL.md"
with urlopen(skill_url) as response:
    skill_content = response.read().decode('utf-8')

store.put(
    namespace=("filesystem",),
    key="/skills/langgraph-docs/SKILL.md",
    value=create_file_data(skill_content),
)

agent = create_deep_agent(
    model="deepseek-v4-flash",
    backend=backend,
    store=store,
    skills=["/skills/"],
    middleware=[CodeInterpreterMiddleware(skills_backend=backend)],
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "LangGraph 是什么？"}]},
    config={"configurable": {"thread_id": "12345"}},
)

print(result["messages"][-1].content[:300])

### 4.3 FilesystemBackend（本地文件系统）

直接使用本地文件系统上的技能目录。适合本地开发和调试。

In [ ]:
# FilesystemBackend 方式：直接读取本地文件系统中的技能

from pathlib import Path

from deepagents import create_deep_agent
from deepagents.backends.filesystem import FilesystemBackend
from langchain_quickjs import CodeInterpreterMiddleware
from langgraph.checkpoint.memory import MemorySaver

# Checkpointer 对于 human-in-the-loop 是必须的
checkpointer = MemorySaver()
root_dir = "/Users/user/{project}"
backend = FilesystemBackend(root_dir=root_dir)

agent = create_deep_agent(
    model="deepseek-v4-flash",
    backend=backend,
    skills=[str(Path(root_dir) / "skills")],  # 指向本地 skills 目录
    interrupt_on={
        "write_file": True,
        "read_file": False,
        "edit_file": True,
    },
    checkpointer=checkpointer,  # 必须配置！
    middleware=[CodeInterpreterMiddleware(skills_backend=backend)],  # 解释器技能需要
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "LangGraph 是什么？"}]},
    config={"configurable": {"thread_id": "12345"}},
)

print(result["messages"][-1].content[:300])

## 5. 参数说明

| 参数 | 类型 | 说明 |
|------|------|------|
| `skills` | `list[str]` | Skill 源路径列表。路径使用正斜杠，相对于 Backend 根目录。省略则不加载技能。 |

更多示例技能请参考 [Deep Agents 示例技能](https://github.com/langchain-ai/deepagents/tree/main/libs/cli/examples/skills)。

---
**参考链接**

- [Skills — LangChain Docs](https://docs.langchain.com/oss/python/deepagents/skills)
- [Agent Skills Specification](https://github.com/langchain-ai/agent-skills-spec)
- [Deep Agents 示例技能](https://github.com/langchain-ai/deepagents/tree/main/libs/cli/examples/skills)
- [Documentation Index](https://docs.langchain.com/llms.txt)